# HospitalIQ – Phase 1: Synthetic Dataset Generation & Data Cleaning
**Healthcare Data Analytics & Patient Segmentation Study | Hyderabad Network**

This notebook demonstrates the synthetic generation of approximately 500,000 healthcare records across 4 Hyderabad hospitals, followed by data cleaning, DOB-to-age conversion, and critical transaction-level GST validation.


In [1]:
import os
import pandas as pd
import numpy as np
from src.data_generator import generate_and_save_all
from src.data_cleaner import clean_all_datasets

# 1. Verify Raw Dataset Generation
raw_dir = os.path.join('data', 'raw')
cleaned_dir = os.path.join('data', 'cleaned')
print('Raw Data Directory:', raw_dir)
print('Cleaned Data Directory:', cleaned_dir)


Raw Data Directory: data\raw
Cleaned Data Directory: data\cleaned


In [2]:
# 2. Inspect Raw Datasets Size
raw_files = ['patients.csv', 'opd_visits.csv', 'admissions.csv', 'lab_tests.csv', 'billing.csv']
total_records = 0
for f in raw_files:
    df = pd.read_csv(os.path.join(raw_dir, f), low_memory=False)
    print(f'{f}: {len(df):,} records')
    total_records += len(df)
print(f'Total Raw Records: {total_records:,}')


patients.csv: 65,000 records
opd_visits.csv: 220,000 records
admissions.csv: 45,000 records
lab_tests.csv: 90,000 records
billing.csv: 81,600 records
Total Raw Records: 501,600


### Critical Billing Accuracy & GST Audit
Under Indian GST regulations, clinical OPD consultations are exempt (0%). Legacy systems introduced:
- Double GST calculation (`Total = Base + 2*GST`)
- Unlawful GST on exempt OPD consultations
- Typographical outliers (₹999,999) and negative refund records

The cleaner uses rule-based matching rather than blind division by 1.18.


In [3]:
# 3. Review Billing Correction Log
corr_log = pd.read_csv(os.path.join(cleaned_dir, 'billing_correction_log.csv'), low_memory=False)
print(f'Total Logged Corrections: {len(corr_log):,}')
print(corr_log['flag'].value_counts())


Total Logged Corrections: 7,869
flag
GST_DOUBLE_COUNTED_CORRECTED     4734
EXEMPT_HEALTHCARE_GST_REMOVED    2562
EXTREME_OUTLIER_KEYING_ERROR      320
NEGATIVE_REFUND_REMOVED           253
Name: count, dtype: int64


In [4]:
# 4. Data Quality Before-and-After Summary
quality_df = pd.read_csv(os.path.join(cleaned_dir, 'data_quality_before_after.csv'))
print(quality_df)


        Table  ...                                  Key Anomaly Fixed
0    Patients  ...        Missing Age imputed via DOB (7,815 records)
1  OPD Visits  ...    Diagnosis standardized (3,357 missing resolved)
2  Admissions  ...  Date inversions fixed (2,783) & Discharges imp...
3   Lab Tests  ...             Date format normalized, costs verified
4     Billing  ...  Duplicates removed (1,600), GST corrected (7,8...

[5 rows x 4 columns]
